# LangGraph 004 — Why LangGraph Exists

Each claim of the lesson, tested. Plain functions stand in for LLM calls, so
the notebook needs only `langgraph` and `langchain-core` — **no API key**.

| Part | What we check |
|---|---|
| A | a loop is one conditional edge: `create_jd` runs 3 times |
| B | LangChain **can** branch (RunnableBranch) but cannot loop back |
| C | retry: `ConnectionError` retried, `TimeoutError` **not**, by default |
| D | after a crash, resume runs earlier steps once |
| E | `interrupt()` pauses for a person; `Command(resume=...)` continues |
| F | a subgraph is one node; every step is recorded |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Control flow

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Hiring(TypedDict):           # the state: every node can read and update it
    jd_version: int
    approved: bool
    posted: bool

verdicts = iter([False, False, True])        # the manager rejects the JD twice

def create_jd(state):
    return {"jd_version": state["jd_version"] + 1}   # return only what changed

def check_approval(state):
    return {"approved": next(verdicts)}

def post_jd(state):
    return {"posted": True}

graph = StateGraph(Hiring)
graph.add_node("create_jd", create_jd)
graph.add_node("check_approval", check_approval)
graph.add_node("post_jd", post_jd)
graph.add_edge(START, "create_jd")
graph.add_edge("create_jd", "check_approval")
graph.add_conditional_edges("check_approval",
                            lambda s: "post_jd" if s["approved"] else "create_jd")   # the loop
graph.add_edge("post_jd", END)

app = graph.compile()
print(app.invoke({"jd_version": 0, "approved": False, "posted": False}))
# {'jd_version': 3, 'approved': True, 'posted': True}   - create_jd ran three times

## Part B — What LangChain has

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

# LangChain CAN branch - RunnableBranch (Generative AI lessons 009 and 011):
route = RunnableBranch(
    (lambda x: x["applicants"] >= 10, RunnableLambda(lambda x: "shortlist")),
    RunnableLambda(lambda x: "modify_jd"),
)
print(route.invoke({"applicants": 25}), route.invoke({"applicants": 3}))
# shortlist modify_jd

# ...but it cannot go BACK to an earlier step. The loop is Python around the chain:
create_jd = RunnableLambda(lambda v: v + 1)
verdicts = iter([False, False, True])
version, approved = 0, False
while not approved:                       # glue code: LangChain cannot see this loop
    version = create_jd.invoke(version)
    approved = next(verdicts)
print("jd_version", version)              # jd_version 3

In [ ]:
assert version == 3
print("same result - but the loop lives outside LangChain")

## Part C — Retry

In [ ]:
from langgraph.types import RetryPolicy

class S(TypedDict):
    x: int

def run(exc):
    calls = {"n": 0}
    def post_jd(state):
        calls["n"] += 1
        if calls["n"] <= 2:                       # fails twice, then works
            raise exc("LinkedIn API unavailable")
        return state
    g = StateGraph(S)
    g.add_node("post_jd", post_jd,
               retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01))
    g.add_edge(START, "post_jd"); g.add_edge("post_jd", END)
    try:
        g.compile().invoke({"x": 1}); ok = True
    except Exception:
        ok = False
    return calls["n"], ok

for exc in (ConnectionError, TimeoutError):
    attempts, ok = run(exc)
    print(f"{exc.__name__:<16} attempts {attempts}  succeeded {ok}")
# ConnectionError  attempts 3  succeeded True
# TimeoutError     attempts 1  succeeded False   <- not retried by default!

# To retry timeouts too, say so:
#   RetryPolicy(retry_on=(ConnectionError, TimeoutError))
# LangChain has retry as well:  some_runnable.with_retry(stop_after_attempt=3)

In [ ]:
assert run(ConnectionError) == (3, True)
assert run(TimeoutError) == (1, False)

# The fix: say which errors to retry
def run_with(exc, retry_on):
    calls = {"n": 0}
    def post_jd(state):
        calls["n"] += 1
        if calls["n"] <= 2:
            raise exc("timeout")
        return state
    g = StateGraph(S)
    g.add_node("post_jd", post_jd,
               retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, retry_on=retry_on))
    g.add_edge(START, "post_jd"); g.add_edge("post_jd", END)
    g.compile().invoke({"x": 1})
    return calls["n"]

assert run_with(TimeoutError, (ConnectionError, TimeoutError)) == 3
print("TimeoutError retried once it is named in retry_on")

## Part D — Resume after a crash

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

class Log(TypedDict):
    log: list

counts, crash = {}, {"armed": True}
STEPS = ["request", "create_jd", "post_jd", "monitor"]

def build(checkpointer):
    g = StateGraph(Log)
    for name in STEPS:
        def node(state, name=name):
            counts[name] = counts.get(name, 0) + 1
            if name == "post_jd" and crash["armed"]:
                crash["armed"] = False
                raise ConnectionError("server went down")
            return {"log": state["log"] + [name]}
        g.add_node(name, node)
    g.add_edge(START, STEPS[0])
    for a, b in zip(STEPS, STEPS[1:]):
        g.add_edge(a, b)
    g.add_edge(STEPS[-1], END)
    return g.compile(checkpointer=checkpointer)

saver = InMemorySaver()
config = {"configurable": {"thread_id": "hire-42"}}   # one thread = one hiring process
try:
    build(saver).invoke({"log": []}, config)
except ConnectionError as e:
    print("crashed:", e, "| next step:", build(saver).get_state(config).next)

print(build(saver).invoke(None, config)["log"])      # None = carry on from the checkpoint
print(counts)
# crashed: server went down | next step: ('post_jd',)
# ['request', 'create_jd', 'post_jd', 'monitor']
# {'request': 1, 'create_jd': 1, 'post_jd': 2, 'monitor': 1}
# Without a checkpointer, starting again runs request and create_jd twice.

In [ ]:
assert counts == {"request": 1, "create_jd": 1, "post_jd": 2, "monitor": 1}

# Now the same crash with no checkpointer: start again from the beginning
counts.clear(); crash["armed"] = True
app = build(None)
try:
    app.invoke({"log": []})
except ConnectionError:
    pass
app.invoke({"log": []})
print(counts)
assert counts["request"] == 2 and counts["create_jd"] == 2

## Part E — Human-in-the-loop

In [ ]:
from langgraph.types import interrupt, Command

class JD(TypedDict):
    jd: str
    decision: str
    posted: bool

def draft(state):
    return {"jd": "Backend engineer, 2-4 years, Python/Django, remote"}

def approve(state):
    answer = interrupt({"question": "Post this JD?", "jd": state["jd"]})   # pause here
    return {"decision": answer}

def post(state):
    return {"posted": state["decision"] == "yes"}

g = StateGraph(JD)
g.add_node("draft", draft); g.add_node("approve", approve); g.add_node("post", post)
g.add_edge(START, "draft"); g.add_edge("draft", "approve")
g.add_edge("approve", "post"); g.add_edge("post", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "jd-1"}}

first = app.invoke({"jd": "", "decision": "", "posted": False}, cfg)
print(first["__interrupt__"][0].value["question"], "| waiting at", app.get_state(cfg).next)
# Post this JD? | waiting at ('approve',)

# ... minutes, hours or days later - nothing is running in the meantime ...
print(app.invoke(Command(resume="yes"), cfg)["posted"])     # True

In [ ]:
cfg2 = {"configurable": {"thread_id": "jd-2"}}
app.invoke({"jd": "", "decision": "", "posted": False}, cfg2)
assert app.invoke(Command(resume="no"), cfg2)["posted"] is False
print("a 'no' is respected: nothing posted")

## Part F — Subgraphs and observability

In [ ]:
class Candidate(TypedDict):
    rounds: list

def step(name):
    return lambda state: {"rounds": state["rounds"] + [name]}

inner = StateGraph(Candidate)                       # the interview: its own workflow
for n in ("technical", "system_design", "culture_fit"):
    inner.add_node(n, step(n))
inner.add_edge(START, "technical"); inner.add_edge("technical", "system_design")
inner.add_edge("system_design", "culture_fit"); inner.add_edge("culture_fit", END)

outer = StateGraph(Candidate)
outer.add_node("shortlist", step("shortlist"))
outer.add_node("interview", inner.compile())       # a whole graph used as one node
outer.add_node("offer", step("offer"))
outer.add_edge(START, "shortlist"); outer.add_edge("shortlist", "interview")
outer.add_edge("interview", "offer"); outer.add_edge("offer", END)

print(outer.compile().invoke({"rounds": []})["rounds"])
# ['shortlist', 'technical', 'system_design', 'culture_fit', 'offer']

In [ ]:
class Count(TypedDict):
    count: int

g = StateGraph(Count)
for n in ("post_jd", "monitor", "shortlist"):
    g.add_node(n, lambda s: {"count": s["count"] + 1})
g.add_edge(START, "post_jd"); g.add_edge("post_jd", "monitor")
g.add_edge("monitor", "shortlist"); g.add_edge("shortlist", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "obs"}}

for update in app.stream({"count": 0}, cfg, stream_mode="updates"):
    print(update)                   # which node ran, and what it changed
# {'post_jd': {'count': 1}}
# {'monitor': {'count': 2}}
# {'shortlist': {'count': 3}}

for snap in reversed(list(app.get_state_history(cfg))):
    print(snap.metadata["step"], snap.next, snap.values)   # 5 checkpoints, oldest first

In [ ]:
history = list(app.get_state_history(cfg))
assert len(history) == 5 and history[0].values == {"count": 3}

## What to take away

- A loop in LangGraph is an edge pointing back. In LangChain it is Python
  around the chain.
- LangChain **can** branch and retry. It cannot loop back, pause or resume.
- The checkpointer is the key: resume, approval and history all rest on it.
- Check the default `retry_on` — timeouts are not retried.

## Exercises

1. Add a "wait 48 hours then re-check applicants" loop to Part A, using a
   counter in the state instead of real time.
2. Make the interview subgraph return a score, and route to "offer" or
   "regret" in the outer graph.
3. Crash the workflow at `monitor` instead. Which counts change?